# Closed-Loop Evaluation (Colab)

This notebook evaluates completed Closed-Loop simulation artifacts (`csv` + `json`) saved in Google Drive.

It uses reusable utilities in `src/eval` to:
- discover and load run artifacts,
- compute method summaries,
- estimate conditional value of surprise beyond risk,
- compute compute-normalized blind-spot discovery efficiency.


## Usage

1. Set your `RUN_TAG`, `SIM_PERSIST_ROOT`, and `EXPORT_DIR`.
2. Run discovery to select merged or shard outputs.
3. Run analysis cells.
4. Export evaluation tables back to Drive.


In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/achyutmorang/waymax-simulation-experiments.git"
REPO_DIR = "/content/waymax-simulation-experiments"

if os.path.exists(REPO_DIR):
    print("Repo exists, fast-forwarding main...")
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "main"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())


In [ ]:
# Mount Google Drive in Colab
import os

if os.environ.get('COLAB_RELEASE_TAG'):
    try:
        from google.colab import drive
        if not os.path.ismount('/content/drive'):
            drive.mount('/content/drive')
        else:
            print('[drive] /content/drive already mounted')
    except Exception as e:
        print('[drive] mount skipped:', e)
else:
    print('[drive] non-Colab environment; skipping mount')


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from src.eval import (
    budget_normalized_efficiency,
    conditional_lift_by_risk_bins,
    deterministic_scenario_split,
    discover_run_prefixes,
    discovery_auc,
    discovery_curve_from_trace,
    load_run_artifacts,
    method_summary,
    paired_effect_significance_table,
    select_default_run_prefix,
)

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 200)


In [ ]:
# ===== User config =====
RUN_TAG = "closedloop_v1"
SIM_PERSIST_ROOT = "/content/drive/MyDrive/waymax_experiments/closedloop_runs/v1"
N_SHARDS = 5
OUT_RUN_TAG = None             # e.g. "closedloop_v1_merged"
PREFER_KIND = "merged"        # "merged" or "shard" or "base"
REQUIRE_TRACE = True

# Statistical checks (surprise advantage over risk)
PRIMARY_TREATMENT = "joint"
PRIMARY_CONTROL = "risk_only"
SIGNIFICANCE_OUTCOMES = ["blind_spot_proxy_hit", "failure_proxy"]
HOLDOUT_FRACTION = 0.20
HOLDOUT_SEED = 17
BOOTSTRAP_SAMPLES = 3000
PERMUTATION_SAMPLES = 5000
SHUFFLE_SAMPLES = 5000
TEST_RANDOM_SEED = 17

# Optional export directory for evaluation outputs
EXPORT_DIR = f"/content/drive/MyDrive/waymax_experiments/closedloop_evaluation/{RUN_TAG}"


In [ ]:
candidates_df = discover_run_prefixes(
    run_tag=RUN_TAG,
    persist_root=SIM_PERSIST_ROOT,
    n_shards=N_SHARDS,
    out_run_tag=OUT_RUN_TAG,
)

display(candidates_df)

selected_run_prefix = select_default_run_prefix(candidates_df, prefer_kind=PREFER_KIND)
print("selected_run_prefix:", selected_run_prefix)


In [ ]:
artifacts = load_run_artifacts(selected_run_prefix, require_trace=REQUIRE_TRACE)

results_df = artifacts.results_df.copy()
trace_df = artifacts.trace_df.copy()
thresholds = artifacts.thresholds

print("results rows:", len(results_df))
print("trace rows:", len(trace_df))
print("threshold keys:", sorted(thresholds.keys())[:12], "...")

display(results_df.head())
if len(trace_df) > 0:
    display(trace_df.head())


In [ ]:
summary_df = method_summary(results_df)
eff_df = budget_normalized_efficiency(results_df)

print("Method summary")
display(summary_df)

print("Budget-normalized efficiency")
display(eff_df)


In [ ]:
# Conditional value of surprise beyond risk:
# treatment (default joint) vs control (default risk_only) within risk bins.

cond_bins_df, cond_overall_df = conditional_lift_by_risk_bins(
    results_df,
    treatment="joint",
    control="risk_only",
    risk_col="risk_sks",
    outcome_col="blind_spot_proxy_hit",
    n_bins=10,
    min_bin_count=5,
    bootstrap_samples=2000,
    bootstrap_seed=17,
)

print("Overall paired effect")
display(cond_overall_df)

print("Risk-bin conditional lift")
display(cond_bins_df)

if len(cond_bins_df) > 0:
    plt.figure(figsize=(8, 4))
    sns.barplot(data=cond_bins_df, x="risk_bin", y="lift", color="#1f77b4")
    plt.xticks(rotation=45, ha='right')
    plt.title("Conditional Blind-Spot Lift (joint - risk_only) by Risk Bin")
    plt.ylabel("Lift in blind_spot_proxy_hit rate")
    plt.xlabel("Risk bin (quantiles)")
    plt.tight_layout()
    plt.show()


## Paired Significance (With Holdout + Negative Control)

This section tests whether `joint` has a statistically credible advantage over `risk_only`:

- paired bootstrap confidence interval on per-scenario deltas,
- paired permutation p-value (one-sided, `joint > risk_only`),
- shuffle negative-control (breaks scenario pairing to check spurious lift),
- deterministic scenario holdout split for claim discipline.


In [ ]:
split_df = deterministic_scenario_split(
    results_df,
    holdout_fraction=HOLDOUT_FRACTION,
    seed=HOLDOUT_SEED,
)

results_eval_df = results_df.merge(split_df, on="scenario_id", how="left")
split_balance_df = (
    results_eval_df.groupby(["eval_split", "method"], as_index=False)
    .agg(n_rows=("scenario_id", "size"), n_scenarios=("scenario_id", "nunique"))
    .sort_values(["eval_split", "method"]) 
    .reset_index(drop=True)
)

print("Split balance")
display(split_balance_df)

sig_tables = []
for outcome in SIGNIFICANCE_OUTCOMES:
    sig_df = paired_effect_significance_table(
        results_eval_df,
        treatment=PRIMARY_TREATMENT,
        control=PRIMARY_CONTROL,
        outcome_col=outcome,
        split_col="eval_split",
        bootstrap_samples=BOOTSTRAP_SAMPLES,
        permutation_samples=PERMUTATION_SAMPLES,
        shuffle_samples=SHUFFLE_SAMPLES,
        seed=TEST_RANDOM_SEED,
    )
    sig_tables.append(sig_df)

paired_significance_df = pd.concat(sig_tables, ignore_index=True) if len(sig_tables) > 0 else pd.DataFrame()

print("Paired significance table")
display(paired_significance_df)

if len(paired_significance_df) > 0:
    primary_holdout = paired_significance_df[
        (paired_significance_df["outcome_col"] == "blind_spot_proxy_hit")
        & (paired_significance_df["split"] == "holdout")
    ]
    if len(primary_holdout) == 1:
        r = primary_holdout.iloc[0]
        print(
            "Primary holdout check:",
            f"mean_delta={r['mean_delta']:.4f},",
            f"CI=[{r['ci_low']:.4f}, {r['ci_high']:.4f}],",
            f"perm_p={r['permutation_pvalue_one_sided']:.4g},",
            f"shuffle_p={r['shuffle_null_p_ge_observed']:.4g}",
        )


In [ ]:
# Compute-normalized discovery efficiency from per-eval trace.

if len(trace_df) == 0:
    print("No per_eval_trace rows found. Re-run with trace export enabled.")
    curve_df = pd.DataFrame()
    auc_df = pd.DataFrame()
else:
    curve_df = discovery_curve_from_trace(
        trace_df=trace_df,
        thresholds=thresholds,
    )
    auc_df = discovery_auc(curve_df)

    display(auc_df)

    plt.figure(figsize=(8, 5))
    sns.lineplot(data=curve_df, x="eval_index", y="discovery_rate", hue="method", marker="o")
    plt.title("Blind-Spot Discovery vs Evaluation Budget")
    plt.xlabel("Evaluation index (compute budget)")
    plt.ylabel("Discovery rate (ever-hit up to eval index)")
    plt.tight_layout()
    plt.show()


In [ ]:
from pathlib import Path

Path(EXPORT_DIR).mkdir(parents=True, exist_ok=True)
prefix_name = Path(selected_run_prefix).name

summary_path = f"{EXPORT_DIR}/{prefix_name}_eval_method_summary.csv"
eff_path = f"{EXPORT_DIR}/{prefix_name}_eval_budget_efficiency.csv"
cond_bins_path = f"{EXPORT_DIR}/{prefix_name}_eval_conditional_lift_bins.csv"
cond_overall_path = f"{EXPORT_DIR}/{prefix_name}_eval_conditional_lift_overall.csv"
split_balance_path = f"{EXPORT_DIR}/{prefix_name}_eval_split_balance.csv"
paired_sig_path = f"{EXPORT_DIR}/{prefix_name}_eval_paired_significance.csv"
curve_path = f"{EXPORT_DIR}/{prefix_name}_eval_discovery_curve.csv"
auc_path = f"{EXPORT_DIR}/{prefix_name}_eval_discovery_auc.csv"

summary_df.to_csv(summary_path, index=False)
eff_df.to_csv(eff_path, index=False)
cond_bins_df.to_csv(cond_bins_path, index=False)
cond_overall_df.to_csv(cond_overall_path, index=False)
if 'split_balance_df' in globals() and isinstance(split_balance_df, pd.DataFrame):
    split_balance_df.to_csv(split_balance_path, index=False)
if 'paired_significance_df' in globals() and isinstance(paired_significance_df, pd.DataFrame):
    paired_significance_df.to_csv(paired_sig_path, index=False)
if 'curve_df' in globals() and isinstance(curve_df, pd.DataFrame):
    curve_df.to_csv(curve_path, index=False)
if 'auc_df' in globals() and isinstance(auc_df, pd.DataFrame):
    auc_df.to_csv(auc_path, index=False)

print("Saved:")
for p in [
    summary_path,
    eff_path,
    cond_bins_path,
    cond_overall_path,
    split_balance_path,
    paired_sig_path,
    curve_path,
    auc_path,
]:
    print(" -", p)


## Notes

- `blind_spot_proxy_hit` is the current Closed-Loop proxy label from simulation outputs.
- The paired-significance table adds holdout split reporting, bootstrap CIs, permutation p-values, and a shuffle negative-control.
- For strict publication claims, report the primary holdout row first, then use other rows as supporting diagnostics.
